<a href="https://colab.research.google.com/github/KarthiksSJEC/QuCardio/blob/main/QuCardio1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🫀 QuCardio – Hybrid Quantum-Classical ECG Classifier
## Phase 1: Classical Pipeline (ResNet50 → PCA → SVM)

**Pipeline at a glance:**
```
ECG Image (224×224 RGB)
    │
    ▼
[ResNet50 – pre-trained]  →  2048-dimensional feature vector
    │
    ▼
[PCA]  →  4 (or 8) core features   ← Quantum-ready bottleneck
    │
    ▼
[Classical SVM]  →  Predicted Class (Normal / Abnormal / MI …)
```

> **Phase 2 (next step – not in this notebook):** Swap SVM for Pegasos QSVC via Qiskit + ZZFeatureMap.

## ⚙️ Step 0 – Install Dependencies

In [ ]:
# These are already available on Colab; uncomment if running elsewhere
# !pip install tensorflow scikit-learn matplotlib seaborn Pillow joblib

## 🔗 Step 1 – Mount Google Drive & Set Dataset Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── UPDATE THIS PATH to where your ECG image folders live ───
# Expected structure:
#   ECG_Dataset/
#       Normal/     ← PNG/JPG images of normal ECGs
#       Abnormal/   ← PNG/JPG images
#       MI/         ← PNG/JPG images (Myocardial Infarction)
#       ...         ← add more classes as needed

DATASET_PATH = '/content/drive/MyDrive/ECG_Dataset'
PCA_COMPONENTS = 4    # set to 4 or 8 per project spec
print(f'Dataset path : {DATASET_PATH}')
print(f'PCA components: {PCA_COMPONENTS}')

## 📦 Step 2 – Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import joblib

import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)
print('All libraries imported successfully ✓')

## 🧠 Step 3 – Load Pre-trained ResNet50 (Feature Extractor)

We strip the top classification layer and keep only the convolutional backbone.
With `pooling='avg'`, each image produces a **2048-dimensional** feature vector.

In [ ]:
print('Loading ResNet50 feature extractor (ImageNet weights, no top layer)...')

feature_extractor = ResNet50(
    weights='imagenet',
    include_top=False,   # remove final 1000-class softmax layer
    pooling='avg'        # global average pooling → output shape (2048,)
)

print(f'✓ ResNet50 loaded.')
print(f'  Input  shape : {feature_extractor.input_shape}')
print(f'  Output shape : {feature_extractor.output_shape}   ← 2048 raw visual features')

## 🖼️ Step 4 – Extract Features from All ECG Images

This cell walks through every class folder, loads each ECG image,
preprocesses it for ResNet50, and stores the 2048-dim feature vector.

In [ ]:
VALID_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}

def load_and_preprocess(img_path, target_size=(224, 224)):
    """Load one ECG image and prepare it for ResNet50."""
    img = Image.open(img_path).convert('RGB')
    img = img.resize(target_size)
    arr = np.array(img, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)    # (1, 224, 224, 3)
    arr = preprocess_input(arr)          # ResNet50 pixel normalisation
    return arr


# ── Walk dataset folder ────────────────────────────────────────────────────
class_names = sorted([
    d for d in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, d))
])
print(f'Found {len(class_names)} classes: {class_names}\n')

all_features, all_labels = [], []

for cls in class_names:
    cls_dir = os.path.join(DATASET_PATH, cls)
    img_files = [
        f for f in os.listdir(cls_dir)
        if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS
    ]
    print(f'  [{cls}] → {len(img_files)} images')

    for fname in img_files:
        fpath = os.path.join(cls_dir, fname)
        try:
            img_arr = load_and_preprocess(fpath)
            feat = feature_extractor.predict(img_arr, verbose=0)  # (1, 2048)
            all_features.append(feat[0])
            all_labels.append(cls)
        except Exception as e:
            print(f'    ⚠ Skipping {fname}: {e}')

# Convert to numpy arrays
X_raw = np.array(all_features)   # (N, 2048)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(all_labels)

print(f'\n✓ Feature matrix shape : {X_raw.shape}')
print(f'✓ Label array shape    : {y.shape}')
print(f'  Classes (encoded)    : {list(zip(label_encoder.classes_, range(len(class_names))))}')

## 🔬 Step 5 – PCA: 2048 → 4 Features

Principal Component Analysis compresses the 2048 ResNet features down to
4 (or 8) dimensions — the **quantum-ready bottleneck**.
Fewer features = fewer qubits needed later.

In [ ]:
pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
X_reduced = pca.fit_transform(X_raw)   # (N, PCA_COMPONENTS)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

print(f'── PCA: 2048 → {PCA_COMPONENTS} components ──')
for i, (ev, cum) in enumerate(zip(explained, cumulative)):
    print(f'  PC{i+1}: {ev*100:.2f}%  (cumulative: {cum*100:.2f}%)')
print(f'\nTotal variance explained: {cumulative[-1]*100:.2f}%')
print(f'Reduced matrix shape    : {X_reduced.shape}')

# ── Scree Plot ─────────────────────────────────────────────────────────────
plt.figure(figsize=(7, 4))
plt.bar(range(1, PCA_COMPONENTS + 1), explained * 100, alpha=0.7,
        color='steelblue', label='Individual')
plt.plot(range(1, PCA_COMPONENTS + 1), cumulative * 100,
         'ro-', label='Cumulative')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained (%)')
plt.title(f'PCA Scree Plot (Top {PCA_COMPONENTS} components of 2048)')
plt.legend()
plt.tight_layout()
plt.show()

## 🤖 Step 6 – Classical SVM Baseline

Train a Support Vector Machine on the 4 PCA features.
This establishes the **classical accuracy baseline** before we swap in the Quantum SVM.

In [ ]:
# ── Train / Test Split ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f'Train samples: {len(X_train)}')
print(f'Test  samples: {len(X_test)}')

# ── Train SVM ─────────────────────────────────────────────────────────────
clf = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42, probability=True)
clf.fit(X_train, y_train)
print('\n✓ SVM trained.')

## 📊 Step 7 – Evaluate & Confusion Matrix

In [ ]:
y_pred = clf.predict(X_test)
y_test_names = label_encoder.inverse_transform(y_test)
y_pred_names = label_encoder.inverse_transform(y_pred)

acc = accuracy_score(y_test, y_pred)

print('=' * 55)
print('  QuCardio – Classical Pipeline Results')
print(f'  PCA dimensions : {PCA_COMPONENTS}')
print(f'  Test Accuracy  : {acc * 100:.2f}%')
print('=' * 55)
print()
print('Classification Report:')
print(classification_report(y_test_names, y_pred_names, target_names=class_names))

# ── Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_test_names, y_pred_names, labels=class_names)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix – Classical SVM (PCA={PCA_COMPONENTS})')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

print('\n✅ Classical pipeline complete!')
print('   Next step → Phase 2: Swap SVM for Pegasos QSVC (Qiskit)')

## 💾 Step 8 – Save Pipeline Artifacts

Save the trained PCA and SVM so they can be loaded in Phase 2
(the Quantum notebook) without re-running feature extraction.

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/QuCardio_Artifacts'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save PCA, SVM, and label encoder
joblib.dump(pca,           os.path.join(SAVE_DIR, 'pca.pkl'))
joblib.dump(clf,           os.path.join(SAVE_DIR, 'svm_classical.pkl'))
joblib.dump(label_encoder, os.path.join(SAVE_DIR, 'label_encoder.pkl'))

# Save raw features (so Phase 2 notebook can load them directly)
np.save(os.path.join(SAVE_DIR, 'X_pca_features.npy'), X_reduced)
np.save(os.path.join(SAVE_DIR, 'y_labels.npy'), y)

print(f'✓ Artifacts saved to {SAVE_DIR}/')
print('  Files: pca.pkl | svm_classical.pkl | label_encoder.pkl')
print('         X_pca_features.npy | y_labels.npy')

## 🔍 Step 9 – Predict on a Single New ECG Image

In [ ]:
# ── UPDATE this path to any ECG image you want to test ────────────────────
TEST_IMAGE_PATH = '/content/drive/MyDrive/ECG_Dataset/Normal/sample_001.png'

img_arr = load_and_preprocess(TEST_IMAGE_PATH)
feat    = feature_extractor.predict(img_arr, verbose=0)   # (1, 2048)
feat_r  = pca.transform(feat)                             # (1, PCA_COMPONENTS)
pred    = clf.predict(feat_r)                             # (1,)
prob    = clf.predict_proba(feat_r).max()
pred_class = label_encoder.inverse_transform(pred)[0]

# Show image
img_display = Image.open(TEST_IMAGE_PATH)
plt.figure(figsize=(5, 4))
plt.imshow(img_display)
plt.axis('off')
plt.title(f'Predicted: {pred_class}  (confidence: {prob*100:.1f}%)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nPipeline output for this ECG:')
print(f'  ResNet50 features : 2048-dim vector')
print(f'  After PCA         : {PCA_COMPONENTS}-dim vector = {feat_r[0].round(4)}')
print(f'  SVM Prediction    : {pred_class}  (confidence {prob*100:.1f}%)')